(statistics_as_gen_model)=
# Understanding regression (and statistics) through predictive modeling
The way we usually report statistics for our data is inferential. Predictive or generative modeling is usually left for machine learning, deep learning, LLMs. For this tutorial predictive and generative mean the same thing. However, many statistics have an underlying predictive or generative model. The ability of your model to predict or generate data is tightly related to how well you can draw inferences from your model. If your model does not fit your data well then the effect sizes and p-values are meaningless. The type of predictions we are related to but not the same as those use for machine learning and other predictive techniques. We want to use predictive modeling to ensure our models fits well so that our inference is correct. I have found that understanding the predictive features of regression models helps you understand how regression models actually work. The nice thing is that we can don't have to do fancy math but, can instead rely on visual comfirmation and intuation which you can later use to build your mathematical intuition of linear regression.
We will use two datasets in this chapter; one is the data used in the current clamp chapter from MSNs and the other is a behavioral dataset that I have found very useful for showing the importance of predictive accuracy in inference. We will start with behavioral data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from lithos import CategoricalPlot, LinePlot

rng = np.random.default_rng(seed=42)

data = pd.read_csv(Path().cwd().parent / "data/stats/regression_as_generative.csv")
data["con_log"] = np.log10(data["consumption"])

First we are going to look at the behavioral data. The dataset is consumption of a substance split across different cohorts and subgroups. Consumption is bounded by 0 and can go to infinity.When data is bounded at 0 and goes to infinity, you have a high chance of having log normal data. There are other distributions that are bounded that you can see in the (distributions chapter)[#distributions]. One thing about this dataset is that it is based on real data but I synthesized. First we will plot the raw data and the log transformed data side by side. One thing you may notice in the raw data is that as the mean value gets larger that variance also gets larger. This is a key sign that your data is not normally distributed by instead a non-normal distribution. In many cases this will be the lognormal distribution. Since this dataset is based on a real dataset the relationship is not perfect. You will notice that log transforming the data rescales the data so that there is non relationship between the mean and variance. This is another good sign that the data is lognormal. Also is does not matter what base log you use to transform the data. The rescaling is the same but the end values will be different.

In [ ]:
fig, ax = plt.subplots(ncols=2, figsize=(10, 4), layout="constrained")
plot = (
    CategoricalPlot(data)
    .grouping(group="Cohort", subgroup="Sex", subgroup_order=["M", "F"])
    .plot_data(y="consumption", ylabel="Consumption", title="Original")
    .jitter(markersize=10, width=0.6)
    .summary(barwidth=0.8, err_func="std")
    .axis(ydecimals=2)
    .axis_format(yminorticks=5)
    .plot(figure=fig, axes=ax[0])
)
plot = (
    CategoricalPlot(data)
    .grouping(group="Cohort", subgroup="Sex", subgroup_order=["M", "F"])
    .plot_data(y="consumption", ylabel="Consumption", title="Log transformed")
    .transform(ytransform="log10", back_transform_yticks=True)
    .jitter(markersize=10, width=0.6)
    .summary(barwidth=0.8, err_func="std")
    .axis(ydecimals=2)
    .axis_format(yminorticks=5)
    .plot(figure=fig, axes=ax[1])
)

Lets look the the relationship between the variance and mean for the both the orginal values and log rescaled values. Also it is important to note that we are transforming the data before we get the mean and standard deviation. We can see that there is a slope for the untransformed data but not the log transformed data.

In [ ]:
f = data.groupby(["Cohort", "Sex"], as_index=False).aggregate(
    consumption_mean=("consumption", "mean"),
    consumption_std=("consumption", "std"),
    con_log_mean=("con_log", "mean"),
    con_log_std=("con_log", "std"),
)
fig, ax = plt.subplots(ncols=2, figsize=(10, 4), layout="constrained")
plot = (
    LinePlot(f)
    .plot_data(
        "consumption_mean",
        "consumption_std",
        title="Original",
        ylabel="Mean",
        xlabel="STD",
    )
    .scatter()
    .fit()
    .plot(figure=fig, axes=ax[0])
)
plot = (
    LinePlot(f)
    .plot_data(
        "con_log_mean",
        "con_log_std",
        title="Log transformed",
        ylabel="Mean",
        xlabel="STD",
    )
    .scatter()
    .fit()
    .axis(xdecimals=3)
    .plot(figure=fig, axes=ax[1])
)

Non-normalized consumption

In [ ]:
formula = "consumption ~ C(Cohort, Diff) + C(Sex, Sum)"
rawmodel = smf.ols(formula, data=data).fit()
print(rawmodel.summary())

Log model

In [ ]:
formula = "con_log ~ C(Cohort, Diff) + C(Sex, Sum)"
logmodel = smf.rlm(formula, data=data, M=sm.robust.norms.HuberT()).fit()
print(logmodel.summary())

In [ ]:
pred_probs = rawmodel.predict(data)
raw_model_predictions = pd.DataFrame(
    {
        "predicted": rng.normal(loc=pred_probs, scale=rawmodel.scale),
        "Cohort": data["Cohort"],
        "Sex": data["Sex"],
        "Tx": data["Tx"],
    }
)
fig, ax = plt.subplots(ncols=2, figsize=(10, 4), layout="constrained")
plot = (
    CategoricalPlot(raw_model_predictions)
    .grouping(group="Cohort", subgroup="Sex", subgroup_order=["M", "F"])
    .plot_data(y="predicted", ylabel="Consumption", title="Generative Data")
    .jitter(markersize=10, width=0.6)
    .summary(barwidth=0.8, err_func="std")
    .axis(ydecimals=2)
    .axis_format(yminorticks=5)
    .plot(figure=fig, axes=ax[0])
)
plot = (
    CategoricalPlot(data)
    .grouping(group="Cohort", subgroup="Sex", subgroup_order=["M", "F"])
    .plot_data(y="consumption", ylabel="Consumption", title="Original Data")
    .jitter(markersize=10, width=0.6)
    .summary(barwidth=0.8, err_func="std")
    .axis(ydecimals=2)
    .axis_format(yminorticks=5)
    .plot(figure=fig, axes=ax[1])
)

In [ ]:
pred_probs = logmodel.predict(data)
logmodel_predictions = pd.DataFrame(
    {
        "consumption": 10 ** rng.normal(loc=pred_probs, scale=logmodel.scale),
        "Cohort": data["Cohort"],
        "Sex": data["Sex"],
        "Tx": data["Tx"],
    }
)
fig, ax = plt.subplots(ncols=2, figsize=(10, 4), layout="constrained")
plot = (
    CategoricalPlot(logmodel_predictions)
    .grouping(group="Cohort", subgroup="Sex", subgroup_order=["M", "F"])
    .plot_data(y="consumption", ylabel="Consumption", title="Generative Data")
    .jitter(markersize=10, width=0.6)
    .summary(barwidth=0.8, err_func="std")
    .axis(ydecimals=2)
    .axis_format(yminorticks=5)
    .plot(figure=fig, axes=ax[0])
)
plot = (
    CategoricalPlot(data)
    .grouping(group="Cohort", subgroup="Sex", subgroup_order=["M", "F"])
    .plot_data(y="consumption", ylabel="Consumption", title="Original Data")
    .jitter(markersize=10, width=0.6)
    .summary(barwidth=0.8, err_func="std")
    .axis(ydecimals=2)
    .axis_format(yminorticks=5)
    .plot(figure=fig, axes=ax[1])
)